# CoT faithfulness under hints and monitoring

Does the chain of thought stay faithful when the model is hinted or told it is monitored?
Model biology / interpretability (Neel Nanda, MATS; Chen et al.; Arcuschin et al.).

Runtime: **T4 GPU**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/zoom-BT/cot-hint-monitoring.git
%cd cot-hint-monitoring

RESULTS = '/content/drive/MyDrive/cot-faithfulness-results'
!mkdir -p "{RESULTS}"

In [ ]:
!pip -q install -r requirements.txt
!python scripts/test_harness.py
!nvidia-smi

In [ ]:
# Pilot: 5 handwritten MCQs x 5 conditions (~25 generations)
!python scripts/run_eval.py \
  --mode pilot \
  --max-questions 5 \
  --output /content/drive/MyDrive/cot-faithfulness-results/generations_pilot.jsonl

In [ ]:
import json
from pathlib import Path

path = Path('/content/drive/MyDrive/cot-faithfulness-results/generations_pilot.jsonl')
for i, line in enumerate(path.open()):
    row = json.loads(line)
    print('=' * 80)
    print(row['condition'], row['question_id'], 'parsed=', row['final_answer'])
    print(row['raw_output'][:600])
    if i >= 4:
        break

In [ ]:
# Main run after the parser looks sane. Resume-safe if the session drops.
!python scripts/run_eval.py \
  --mode main \
  --output /content/drive/MyDrive/cot-faithfulness-results/generations.jsonl \
  --no-cot-baseline

In [ ]:
!python scripts/metrics.py \
  --input /content/drive/MyDrive/cot-faithfulness-results/generations.jsonl \
  --summary-out /content/drive/MyDrive/cot-faithfulness-results/metrics_summary.csv \
  --manual-sample-out /content/drive/MyDrive/cot-faithfulness-results/manual_label_sample.csv \
  --figures-dir /content/drive/MyDrive/cot-faithfulness-results/figures

In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.read_csv('/content/drive/MyDrive/cot-faithfulness-results/metrics_summary.csv'))
display(Image(filename='/content/drive/MyDrive/cot-faithfulness-results/figures/hint_compliance_by_condition.png'))
display(Image(filename='/content/drive/MyDrive/cot-faithfulness-results/figures/silent_shift_by_condition.png'))